Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

Data Preprocessing  

B1

1 -> Loading dataset

In [ ]:
df = pd.read_csv("Loan_default_dataset.csv")
print("Initial dataset shape:", df.shape)

2 -> Basic Data Cleaning

In [ ]:
df.drop_duplicates(inplace=True)

if 'LoanID' in df.columns:
    df.drop('LoanID', axis=1, inplace=True)

3 -> Identify Missing Values

In [ ]:
print("\nMissing values before imputation:")
print(df.isnull().sum())

4 -> Handling Missing Data

In [ ]:
numeric_features = ['Age', 'Income', 'LoanAmount', 'CreditScore', 
                    'MonthsEmployed', 'NumCreditLines', 'InterestRate', 
                    'LoanTerm', 'DTIRatio']
categorical_features = ['Education', 'EmploymentType', 'MaritalStatus', 
                        'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']

binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

num_imputer = SimpleImputer(strategy='median')
df[numeric_features] = num_imputer.fit_transform(df[numeric_features])

cat_cols = list(set(categorical_features) - set(binary_cols))
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("\nMissing values after imputation:")
print(df.isnull().sum())

5 -> Feature Engineering

In [ ]:
df['LoanToIncomeRatio'] = df['LoanAmount'] / df['Income']
df['FinancialRiskScore'] = df['CreditScore'] - df['LoanToIncomeRatio'] + (df['MonthsEmployed'] / 12)

numeric_features.extend(['LoanToIncomeRatio', 'FinancialRiskScore'])

6 -> Categorical Encoding

In [ ]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

7 -> Scaling Numerical Features

In [ ]:
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

for col in numeric_features:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower, upper)

In [ ]:
print("\nFinal dataset shape:", df.shape)
print("\nFirst few rows of the preprocessed data:")
print(df.head())

df.to_csv("loan_default_preprocessed.csv", index=False)

Model Training  

B2

In [ ]:
df = pd.read_csv("loan_default_preprocessed.csv")
print("Dataset shape:", df.shape)

X = df.drop("Default", axis=1)
y = df["Default"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

In [ ]:
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

B3

In [ ]:
xgb_model.fit(X_train, y_train)